# 01 - Pirate LoRA toy (SmolLM2-360M-Instruct)

Learn LoRA by doing: take an open-weight instruct model, teach it pirate speak with a Low-Rank Adapter, and *see* the difference.

Pipeline in this notebook:

1. **Env check** (this section) - torch/HIP, precision decision
2. **Data** (issue #2) - `quill-voice/pirate`, 90/10 split, 5 holdout prompts
3. **Before + r=8 run** (issue #3) - baseline samples, first adapter
4. **r=32 run + compare + merge** (issue #4) - rank sweep, curves, `merge_and_unload`

How we work: read the explanation above each code cell *before* running it. If a cell surprises you, stop and ask - that surprise is the lesson.

## 0. Environment: why torch comes from AMD, not PyPI

PyPI `torch` wheels are CUDA/CPU-only - on our RX 7900 XTX (gfx1100) `torch.cuda.is_available()` would be `False` with them. So we install torch from AMD's wheel index (pinned in `scripts/install-rocm-torch.sh`). Run the cell below to prove the venv sees the GPU. If it prints the CPU fallback instead, re-run that script - everything after this cell assumes the printed precision decision (`bf16` on HIP, `fp32` on CPU).

In [ ]:
import platform
import torch
import matplotlib

print(f"python: {platform.python_version()}")
print(f"torch: {torch.__version__}")
hip = torch.cuda.is_available()
print(f"hip_available: {hip}")
if hip:
    print(f"device_count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"device[{i}]: {torch.cuda.get_device_name(i)}")
    use_bf16 = torch.cuda.is_bf16_supported()
else:
    print("device: CPU (fallback - see scripts/install-rocm-torch.sh)")
    use_bf16 = False
print(f"matplotlib: {matplotlib.__version__}")
print("precision decision:", "bf16" if use_bf16 else "fp32")

## 0b. Plotting smoke test

Later sections plot train/val loss, learning-rate schedule, and grad-norm with `matplotlib`. This cell proves plotting works in your setup - expect a sine wave, nothing more.

In [ ]:
import math
import matplotlib.pyplot as plt

xs = [i / 10 for i in range(100)]
ys = [math.sin(x) for x in xs]
plt.figure()
plt.plot(xs, ys)
plt.title("smoke test - real loss curves arrive in issues #3/#4")
plt.xlabel("step")
plt.ylabel("sin(step)")
plt.show()

## 1. Data: pirate rows, frozen split

`quill-voice/pirate` is 797 `(instruction -> pirate output)` rows - e.g. "Write a Python function..." answered in pirate voice. Two decisions matter here:

1. **Holdouts are excluded from train AND val.** Five prompts are frozen in `data/holdouts.json` as pure demo material. If they stayed in train, the "after" samples in issues #3/#4 would partly be memorization, not style learning. Style is global so 5 rows change nothing measurable - but the habit (never eval on train) is the point.
2. **Split is seeded (42) and re-asserted.** Both LoRA runs (r=8, r=32) must see identical train/val, or the comparison is meaningless. `scripts/prepare_data.py` froze the counts in `data/split.json`; the cell below re-does the split and asserts equality - rerun it any time to prove reproducibility.

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
split_info = json.loads((DATA_DIR / "split.json").read_text())
holdouts = json.loads((DATA_DIR / "holdouts.json").read_text())
print(split_info)

ds = load_dataset(split_info["dataset"], split="train")
assert len(ds) == split_info["total_n"], "dataset changed upstream!"
by_instr = {r["instruction"] for r in ds}
assert all(h in by_instr for h in holdouts), "a holdout is not in the dataset verbatim"

HOLD = set(holdouts)
pool = ds.filter(lambda r: r["instruction"] not in HOLD)
parts = pool.train_test_split(test_size=0.1, seed=split_info["seed"])
train, val = parts["train"], parts["test"]
assert (len(train), len(val)) == (split_info["train_n"], split_info["val_n"])
print(f"train: {len(train)} | val: {len(val)} | holdouts: {len(holdouts)} (in neither)")
print("holdout prompts:")
for h in holdouts:
    print(" -", h)

### Tokenization: ChatML + assistant-only labels

Two steps, kept visible on purpose: **format** (`apply_chat_template`, ChatML with `<|im_start|>`) then **tokenize**. Note `transformers>=5` returns a `BatchEncoding` dict here, not a bare id list - hence `["input_ids"]`.

The key idea is `labels`: we mask every token up to the assistant's answer with `-100` (PyTorch's "ignore this position" id). We only want to teach *pirate answers*, not pirate questions - without masking, the loss would also reward memorizing user prompts. The prefix boundary is exact (verified: prefix tokenizes identically inside the full sequence on all 797 rows), and nothing exceeds 512 tokens, so `truncation=True` is pure safety.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")

def format_row(instruction, output):
    return tok.apply_chat_template(
        [{"role": "user", "content": instruction},
         {"role": "assistant", "content": output}],
        tokenize=False,
    )

def tokenize_row(instruction, output, max_length=512):
    prefix = tok.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False, add_generation_prompt=True,
    )
    full = format_row(instruction, output)
    input_ids = tok(full, truncation=True, max_length=max_length)["input_ids"]
    pre_len = len(tok(prefix, truncation=True, max_length=max_length)["input_ids"])
    pre_len = min(pre_len, len(input_ids))
    labels = [-100] * pre_len + input_ids[pre_len:]
    return input_ids, labels

print("3 tokenized samples (instruction | ids | supervised positions):")
for i in [0, 1, 2]:
    r = train[i]
    ids, labels = tokenize_row(r["instruction"], r["output"])
    kept = sum(l != -100 for l in labels)
    print(f"[{i}] {r['instruction'][:60]!r} | ids={len(ids)} | supervised={kept}")
    assert kept > 0 and len(ids) <= 512

In [ ]:
lens = [len(tokenize_row(r["instruction"], r["output"])[0]) for r in train]
lens.sort()
print(f"train tokenized length: min={lens[0]} median={lens[len(lens)//2]} max={lens[-1]} (cap 512)")
print(f"rows that would truncate: {sum(1 for l in lens if l >= 512)} - headroom confirmed")

## Next: before-samples + r=8 run (issue #3)

Data frozen and understood. Next: decode the 5 holdouts on the *base* model (the "before"), then train the first adapter (r=8, alpha 16, 1 epoch) and watch train/val loss move.